<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/SVD_Circuits_Demo.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SVD Circuits Demo

Component-level circuit analysis (`direct_logit_attribution`, path patching, head detectors) treats an attention head as an atomic unit. But a single head can superpose several subfunctions on distinct low-rank directions of its QK and OV maps. This notebook decomposes a head's QK ($W_Q W_K^\top$) and OV ($W_V W_O$) maps into orthogonal singular directions, and causally gates every claimed subfunction by patching activations along that direction.

Based on Areeb Ahmad, Abhinav Joshi, Ashutosh Modi, "Beyond Components: Singular Vector-Based Interpretability of Transformer Circuits", [arXiv 2511.20273](https://arxiv.org/abs/2511.20273).

## Setup (Ignore)

In [1]:
# NBVAL_IGNORE_OUTPUT
# Janky code to do different setup when run in a Colab notebook vs VSCode
import os

DEVELOPMENT_MODE = False
IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"
try:
    import google.colab

    IN_COLAB = True
    print("Running as a Colab notebook")
except ImportError:
    IN_COLAB = False
    print("Running as a Jupyter notebook - intended for development only!")
    DEVELOPMENT_MODE = True
    from IPython import get_ipython

    ipython = get_ipython()
    ipython.run_line_magic("load_ext", "autoreload")
    ipython.run_line_magic("autoreload", "2")

if IN_COLAB or IN_GITHUB:
    %pip install transformer_lens

Running as a Jupyter notebook - intended for development only!


In [2]:
# NBVAL_IGNORE_OUTPUT
import torch

from transformer_lens.model_bridge import TransformerBridge
from transformer_lens.tools.analysis.svd_circuits import (
    decompose_head,
    patch_along_directions,
    project_activations,
    vocab_readout,
)

MODEL_ID = "gpt2"

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TransformerBridge.boot_transformers(MODEL_ID, dtype=torch.float32, device=device)
# The vocab readout projects through the final LayerNorm folded into W_U, so the
# bridge has to be in compatibility mode before any readout is taken.
model.enable_compatibility_mode()
model.eval()
print(f"loaded {MODEL_ID}: {model.cfg.n_layers} layers, d_model={model.cfg.d_model}, {device=}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

loaded gpt2: 12 layers, d_model=768, device='cpu'


## 1. Decompose a name-mover head

Layer 9 head 9 is the canonical name-mover head in GPT-2-small's indirect-object-identification (IOI) circuit. Two weight maps describe what it does:

- **QK** $= W_Q W_K^\top$ decides which residual-stream directions the head compares when forming its attention pattern.
- **OV** $= W_V W_O$ decides what the head writes back into the residual stream.

Both are rank at most $d_{\text{head}} = 64$, living inside a $d_{\text{model}} = 768$ space. `FactoredMatrix` computes their SVD without ever materialising the $768 \times 768$ product.

In [3]:
PROMPT = "When Mary and John went to the store, John gave a drink to"
LAYER, HEAD = 9, 9

decomposition = decompose_head(model, layer=LAYER, head=HEAD, which=("QK", "OV"))
ov = decomposition.OV

print(f"{'idx':>4} {'sigma':>10} {'sigma/sigma_max':>16} {'degenerate':>11} {'null':>6}")
for row in ov.rank_report[:12]:
    print(
        f"{row.idx:>4} {row.sigma:>10.4f} {row.sigma_ratio:>16.4f} "
        f"{str(row.is_degenerate):>11} {str(row.is_null):>6}"
    )

 idx      sigma  sigma/sigma_max  degenerate   null
   0     8.4096           1.0000       False  False
   1     8.2600           0.9822       False  False
   2     8.1019           0.9634       False  False
   3     8.0099           0.9525        True  False
   4     7.9312           0.9431        True  False
   5     7.8743           0.9364        True  False
   6     7.8141           0.9292        True  False
   7     7.7938           0.9268        True  False
   8     7.7175           0.9177        True  False
   9     7.6490           0.9096        True  False
  10     7.5680           0.8999        True  False
  11     7.5578           0.8987        True  False


### Reading the rank report

Near-equal consecutive singular values define a subspace whose basis is only defined up to a rotation, so individual directions inside such a block are not attributable. Numerically null directions are arbitrary vectors from the map's null space, and are equally unusable.

The tool refuses per-direction attribution inside a degenerate block and reports the block as a subspace instead. This is a guard, not a footnote: without it, "direction 3 is the surname subfunction" is a claim about an arbitrary basis choice.

## 2. What tokens does each direction move?

In [4]:
readout = vocab_readout(model, ov, k=10)

# readout is [d_vocab, k]: column i holds direction i's projection onto every token,
# so the top tokens for a direction are the largest-magnitude entries of that column.
for i in range(5):
    top = torch.topk(readout[:, i].abs(), 10).indices
    tokens = [model.tokenizer.decode([int(t)]) for t in top]
    print(f"direction {i}: {tokens}")


direction 0: ['Sav', 'AV', 'av', 'adh', 'avour', 'aw', ' Gaw', 'tein', ' Sav', 'amac']
direction 1: ['NI', 'ANI', 'irin', 'Ir', ' NI', 'tro', ' Irwin', 'iq', 'ani', ' Ir']
direction 2: [' Lindsay', ' McKenna', 'etr', ' 339', 'Lind', ' copper', ' seiz', ' Ack', 'gae', ' Instr']
direction 3: [' Bro', 'cn', ' 159', 'ケ', '159', ' Nik', '161', ' Conor', 'CN', ' Gol']
direction 4: [' Wilkinson', ' Kry', ' Luxem', ' Rey', 'wu', 'osa', 'Wil', 'ulative', 'Hu', ' Ned']


A clean vocab readout is a hypothesis, not evidence. `SVDInterpreter`'s own docstring warns that singular directions are not reliably interpretable, and a direction can project onto plausible tokens while carrying no causal role in the model's behaviour. The next two sections test that.

## 3. How much does each direction fire on a prompt?

In [5]:
projection = project_activations(model, ov, PROMPT)

print("prompt tokens:", projection.str_tokens)
print()
print(f"{'token':>10} " + " ".join(f"{f'dir{i}':>8}" for i in range(5)))
for pos, token in enumerate(projection.str_tokens):
    coeffs = " ".join(f"{abs(float(projection.coefficients[pos, i])):>8.3f}" for i in range(5))
    print(f"{token:>10} {coeffs}")

prompt tokens: ['<|endoftext|>', 'When', ' Mary', ' and', ' John', ' went', ' to', ' the', ' store', ',', ' John', ' gave', ' a', ' drink', ' to']

     token     dir0     dir1     dir2     dir3     dir4
<|endoftext|>    0.341    0.203    0.173    0.192    0.570
      When    0.336    0.206    0.189    0.215    0.561
      Mary    0.308    0.213    0.194    0.241    0.605


       and    0.198    0.367    0.507    0.906    1.171
      John    0.261    0.185    0.182    0.228    0.569
      went    0.342    0.202    0.173    0.186    0.559
        to    0.266    0.175    0.163    0.255    0.555
       the    0.339    0.216    0.197    0.216    0.576
     store    0.311    0.235    0.225    0.233    0.574
         ,    3.464    0.040    0.647    3.340    2.227
      John    0.273    0.174    0.159    0.133    0.493
      gave    5.595    1.413    2.725    7.109    5.703
         a    0.253    0.270    0.217    0.212    0.594
     drink    0.290    0.188    0.242    0.245    0.592
        to    7.314    1.745    3.408    9.126    7.349


## 4. The causal gate

This is the mandatory step that distinguishes the tool from a static SVD readout. `patch_along_directions` reconstructs the head's output onto a chosen singular subspace and reports:

- `delta_metric`: patched metric minus original metric.
- `baseline_delta_metric`: the mean magnitude over random same-width subspaces drawn inside the head's own OV span, so the control is an arbitrary subspace of *this head's* output rather than an unrelated residual-stream direction.
- `gated`: whether the retained subspace passed the causal test.

The meaning of `gated` depends on the mode:

- With `keep=[i]` (retain only direction $i$), `gated` means `abs(delta_metric) < baseline`: the direction alone reconstructs the head's behaviour.
- With `ablate=[i]` (zero direction $i$), `gated` means `abs(delta_metric) > baseline`: removing it matters.

`keep=S` and `ablate=complement(S)` resolve to the same retained set, which is why a single "moved more than baseline" test cannot answer both questions.

In [6]:
mary_token = model.to_single_token(" Mary")
john_token = model.to_single_token(" John")
metric = lambda logits: float(logits[0, -1, mary_token] - logits[0, -1, john_token])

# Sweep every attributable direction, not just the first few: the gate is a
# comparison against a random control, so which directions pass is the result.
print(f"{'dir':>4} {'sigma':>9} {'delta':>10} {'baseline':>10} {'gated':>6}")
for row in ov.rank_report:
    if row.is_degenerate:
        continue
    result = patch_along_directions(
        model, ov, PROMPT, metric, keep=[row.idx], rng=torch.Generator().manual_seed(0)
    )
    print(
        f"{row.idx:>4} {row.sigma:>9.4f} {result.delta_metric:>10.4f} "
        f"{result.baseline_delta_metric:>10.4f} {str(result.gated):>6}"
    )


 dir     sigma      delta   baseline  gated


   0    8.4096     0.1543     0.1272  False


   1    8.2600     0.1407     0.1272  False


   2    8.1019     0.1322     0.1272  False


  23    6.8420     0.1932     0.1272  False


...

  55    5.0464     0.0776     0.1272   True


  56    4.9738     0.1140     0.1272   True


  59    4.7041     0.1307     0.1272  False


  60    4.6103     0.1357     0.1272  False


  61    4.4351     0.1718     0.1272  False


In [7]:
# In ablate mode `gated` means the opposite thing: the direction was load-bearing.
print(f"{'dir':>4} {'sigma':>9} {'delta':>10} {'baseline':>10} {'gated':>6}")
for row in ov.rank_report:
    if row.is_degenerate:
        continue
    result = patch_along_directions(
        model, ov, PROMPT, metric, ablate=[row.idx], rng=torch.Generator().manual_seed(0)
    )
    print(
        f"{row.idx:>4} {row.sigma:>9.4f} {result.delta_metric:>10.4f} "
        f"{result.baseline_delta_metric:>10.4f} {str(result.gated):>6}"
    )


 dir     sigma      delta   baseline  gated


   0    8.4096    -0.0122     0.0411  False


   1    8.2600    -0.0155     0.0411  False


   2    8.1019    -0.0139     0.0411  False


  23    6.8420    -0.0576     0.0411   True


...

  55    5.0464     0.0404     0.0411  False


  56    4.9738     0.0252     0.0411  False


  59    4.7041     0.0022     0.0411  False


  60    4.6103    -0.0162     0.0411  False


  61    4.4351    -0.0324     0.0411  False


## 5. What this does and does not show

- **Single head, single model.** Everything above is layer 9 head 9 of `gpt2-small`. Nothing here assembles a multi-head circuit.
- **Mechanism, not inventory.** The paper's headline subfunction taxonomy is scale- and model-dependent. `gpt2-small` reproduces the *mechanism* (a name-mover splitting into causally distinct directions), not necessarily the paper's exact subfunction inventory at larger scale.
- **Seed sensitivity.** Directions whose `delta_metric` sits near the baseline threshold can gate either way across random-control draws. The `rng` seed is pinned above so the tables reproduce; a verdict close to the threshold should be read as borderline, not settled.
- **Weight-space plus one intervention.** The decomposition is closed-form and training-free, and the gate is a single-subspace intervention. Neither is a full circuit-level causal claim, and a passing gate does not by itself name what the direction computes.
- **No automated labelling.** Direction names come from eyeballing the vocab readout, which is exactly the step the causal gate exists to discipline.